**Figure 4: Tail Latency Scaling with Trajectory Length.** Per-call and per-trajectory
p50/p95 at 54/64/96 calls for the two AgentTX modes. Results suggest that the median
per-call latency is stable as trajectories grow, while the p95 gap between full and
no-trace stays roughly constant -- the read-trace tax neither amortizes nor
compounds with length, which pins it as a per-call, capture-side cost.


In [ ]:
# ipython -c "%run plot_tail_scaling.ipynb"
# Shared USENIX plotting convention (FAST/OSDI camera-ready).
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, in cm
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['font.family'] = 'Nimbus Roman'
pd.options.display.max_columns = None

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

plt.rcParams['axes.grid.axis'] = 'both'
df = pd.read_csv(RESULTS / 'motivation_tail_scaling.csv')
lengths = sorted(df['length'].unique())
modes = ['agenttx_without_read_tracing', 'agenttx_full']
labels = ['AgentTX no-trace', 'AgentTX full']
colors = ['#2b2d42', '#ef233c']
markers = ['x', '>']
line_types = ['--', '-']

def series(metric, mode):
    return [float(df[(df['length'] == length) & (df['mode'] == mode)][metric].iloc[0]) for length in lengths]

fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(5.8)))
metrics = [
    ('step_p50_ms', 'Per-call p50 (ms)', '(a) Median call latency', '{:.0f}'),
    ('step_p95_ms', 'Per-call p95 (ms)', '(b) Tail call latency', '{:.0f}'),
    ('run_p50_ms', 'Trajectory p50 (ms)', '(c) Median trajectory latency', '{:.0f}'),
    ('run_p95_ms', 'Trajectory p95 (ms)', '(d) Tail trajectory latency', '{:.0f}'),
]
line_handles = []
for plot_id, (metric, ylabel, title, formatter) in enumerate(metrics):
    ax = plt.subplot(2, 2, plot_id + 1)
    for mode_id, mode in enumerate(modes):
        values = series(metric, mode)
        line, = ax.plot(lengths, values, color=colors[mode_id], marker=markers[mode_id], linestyle=line_types[mode_id], linewidth=0.7, markersize=3, markeredgewidth=0.6, label=labels[mode_id])
        if plot_id == 0:
            line_handles.append(line)
        for x_value, y_value in zip(lengths, values):
            ax.annotate(formatter.format(y_value), (x_value, y_value), textcoords='offset points', xytext=(0, 4 if mode_id == 0 else -9), ha='center', fontsize=5)
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Trajectory length (# calls)', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(lengths)
    ax.tick_params(bottom=False, top=False, left=False, right=False)
    ax.tick_params(axis='both', labelsize=8)
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(0.5)

fig.legend(line_handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=False, columnspacing=1.0, handletextpad=0.2, handlelength=1.5, fontsize=8)
plt.tight_layout(pad=0.4, rect=[0.04, 0.0, 0.99, 0.92])
plt.savefig(FIGDIR / 'FIG-Motivation-Tail-Scaling.pdf', bbox_inches='tight', pad_inches=0)
plt.savefig(FIGDIR / 'FIG-Motivation-Tail-Scaling.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

p95_full = np.asarray(series('step_p95_ms', 'agenttx_full'))
p95_nt = np.asarray(series('step_p95_ms', 'agenttx_without_read_tracing'))
print(f"p95 read-trace gap across lengths: {[f'{v:.2f}x' for v in (p95_full / p95_nt)]}")
